# PlainMed — GPU benchmark

Measures what MedGemma actually costs on this GPU:

1. **Latency** per report
2. **Peak GPU memory** — which card you can rent
3. **Validator rejection rate** — how much quality a quantization level costs

That third number is the one worth having. It is the share of generated
statements that PlainMed's validator *refused* to show, because they cited
a line that does not exist, contained a number absent from that line, or
made a claim the report does not support. It is a direct quality signal,
and it only exists because the validator does.

Runs on the **free Colab T4**. About 30 minutes end to end.

---

### Before you start

**1. Accept the model licence.** MedGemma is gated. Open
[huggingface.co/google/medgemma-1.5-4b-it](https://huggingface.co/google/medgemma-1.5-4b-it),
sign in, and accept. Approval is normally instant.

**2. Create a read token** at Hugging Face → Settings → Access Tokens.

**3. Add it to Colab.** Click the 🔑 key icon in the left sidebar →
*Add new secret* → name it `HF_TOKEN`, paste the token, and turn
**Notebook access** on. Do not paste the token into a cell.

**4. Turn on the GPU.** Runtime → Change runtime type → **T4 GPU** → Save.
Forgetting this is the most common reason the benchmark fails.


## 1. Confirm a GPU is attached

If this errors, the runtime type is not set to GPU.


In [ ]:
!nvidia-smi


## 2. Get the code


In [ ]:
!git clone https://github.com/marisprabhu/plainmed.git
%cd plainmed


## 3. Install

Only the `llm` extra — the benchmark runs on text samples and needs no OCR,
so skipping the `gpu` extra avoids a long PaddlePaddle install.


In [ ]:
!pip install -q -e ".[llm]" bitsandbytes


## 4. Download the weights

About 9 GB; a few minutes. Fails with a 401 if you have not accepted the
licence, or if `HF_TOKEN` is missing from Colab secrets.


In [ ]:
from google.colab import userdata
from huggingface_hub import snapshot_download

snapshot_download(
    "google/medgemma-1.5-4b-it",
    local_dir="models/medgemma-1.5-4b-it",
    token=userdata.get("HF_TOKEN"),
)


## 5. Benchmark at 4-bit

Start here. The T4 is a Turing card with **no bf16 support**, and 4-bit NF4
is also the configuration that makes a cheap 16 GB GPU viable — which is the
claim the architecture rests on.


In [ ]:
!PLAINMED_BACKEND=medgemma PLAINMED_QUANTIZATION=4bit \
  python scripts/benchmark_model.py


## 6. Benchmark at 8-bit, and compare

If 4-bit pushes the rejection rate much above 20%, quality has degraded past
usefulness and 8-bit or a larger card is the right answer. Recording *both*
is more convincing than reporting the better one.


In [ ]:
!PLAINMED_BACKEND=medgemma PLAINMED_QUANTIZATION=8bit \
  python scripts/benchmark_model.py


## 7. Optional — full precision

Likely to run out of memory on a 16 GB T4, and slow if it does not, since
Turing lacks bf16. An honest OOM is a legitimate result: it is the evidence
that quantization is doing real work.


In [ ]:
!PLAINMED_BACKEND=medgemma PLAINMED_QUANTIZATION=auto \
  python scripts/benchmark_model.py


---

## What to do with the output

Copy the three numbers onto the evidence slide, and say which GPU produced
them. A judge's first question is always *on what hardware* — a number
without a card attached to it is not a measurement.

**Screen-record cell 5 running.** A terminal producing real numbers is worth
more than the same numbers typed onto a slide.

### Reproducing the safety claims

These need no GPU and run in seconds:


In [ ]:
!python -m pytest -q
!python scripts/offline_check.py
!python scripts/retention_check.py
!python scripts/deident_check.py


---

> **PlainMed is a research prototype.** It is not a medical device, is not
> clinically validated, and its glossary has not been reviewed by a clinician.
> The samples used here are synthetic. Never put real patient data in a
> notebook.
